# 06. dit.py + Flow Matching: что именно учится

DiT — сеть скорости движения в латентном пространстве. Сделаем настоящий forward уменьшенной официальной архитектуры и отдельное учебное обучение velocity. Не обучаем 2B модель на ноутбуке.


[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladislav-vasilenko/generative-ai-research/blob/main/kandinsky-5/notebooks/06_dit_and_flow_matching.ipynb)

### Подготовка Google Colab

Первая ячейка загружает код курса и устанавливает зависимости. Большие веса моделей она не скачивает. Для учебных опытов достаточно CPU; полная генерация в notebook 10 требует NVIDIA GPU. Если Colab попросит перезапуск сессии после установки, перезапустите и снова выполните ячейки сверху вниз.


In [ ]:
# Этот шаг нужен только в Google Colab; локально используется установленное окружение.
from pathlib import Path
import sys, subprocess
if 'google.colab' in sys.modules:
    checkout=Path('/content/generative-ai-research')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    course=checkout/'kandinsky-5'
    if not (course/'labkit').is_dir():
        # Пока PR не влит, берём его ветку. После merge используется main.
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin','kandinsky-5-labs'],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
    course=checkout/'kandinsky-5'
    expected={'torch':'2.8.0','transformers':'4.56.2','diffusers':'0.35.1'}
    loaded={name:str(getattr(sys.modules[name],'__version__','')) for name in expected if name in sys.modules}
    subprocess.run([sys.executable,'-m','pip','install','-r',str(course/'requirements-labs.txt')],check=True)
    if any(version.split('+')[0]!=expected[name] for name,version in loaded.items()):
        raise RuntimeError('Библиотека уже была импортирована до установки. Перезапустите сессию Colab и выполните notebook с первой ячейки.')
    if str(course) not in sys.path: sys.path.insert(0,str(course))
    print('Course:',course)
    print('Git revision:',subprocess.check_output(['git','-C',str(checkout),'rev-parse','HEAD'],text=True).strip())
else:
    print('Локальный запуск: зависимости устанавливаются по README.')


### Перед началом

Запускайте ячейки сверху вниз. Для каждой операции сначала прочитайте пояснение, затем выполните код и сравните вывод с ожидаемым. Установка описана в README; сетевых загрузок в учебном режиме нет.


In [ ]:
from pathlib import Path
import sys, os
# В Colab загрузите архив KandinskyLab-colab.zip через панель Files и распакуйте
# ячейкой из README. Здесь ищем корень курса, не меняя рабочую директорию.
candidates=[Path.cwd(), *Path.cwd().parents, Path('/content/KandinskyLab'), Path('/content/generative-ai-research/kandinsky-5')]
ROOT=next((p for p in candidates if (p/'labkit').is_dir()), None)
if ROOT is None: raise RuntimeError('Сначала распакуйте архив курса; инструкции в README.')
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
os.environ.setdefault('MPLCONFIGDIR',str(ROOT/'artifacts/matplotlib'))
import torch, numpy as np, matplotlib.pyplot as plt
from labkit.common import SOURCE, REVISION, device_info, source_excerpt, release, shifted_schedule
# Маленькие эксперименты по умолчанию CPU: быстро и одинаково в Mac/Colab.
# Для отдельной проверки Metal смените LAB_DEVICE на mps.
LAB_DEVICE='cpu'
torch.manual_seed(42)
torch.set_num_threads(2)
print(device_info())


## Часть A. Что значит Flow Matching «внутри DiT»

**DiT — архитектура функции** $v_\theta(x_t,t,c)$; **Flow Matching — способ её обучения**; **Euler — алгоритм интегрирования после обучения**. В `dit.py` нет optimizer или FM loss. Сеть получает промежуточный латент, время и conditioning, а возвращает velocity. Построение training пары и loss находится в trainer, а ODE шаг — в `generation_utils.py`.

В сохранённом репозитории есть inference implementation, а полный pretraining trainer не опубликован. Ниже воспроизводим прозрачную conditional straight-path FM постановку, согласованную со знаком upstream sampler, и отдельно показываем подтверждённый execution path. Не приписываем свой uniform time sampling всей истории обучения/дистилляции Kandinsky.

### A1. Непрерывная картина: множество частиц движется по полю

Представьте облако точек noise. В каждой точке сеть задаёт стрелку; интегратор многократно двигает частицы. Совокупность траекторий преобразует одно распределение в другое. ODE описывает отдельную траекторию:

$$\frac{dx_t}{dt}=v_\theta(x_t,t,c).$$

На уровне плотности это соответствует continuity equation $\partial_t p_t+\nabla\cdot(p_t v_t)=0$: масса перераспределяется, а не исчезает. Для обычной генерации не нужно вычислять плотность или divergence; достаточно velocity forward.

### A2. Соглашение времени именно в нашем Kandinsky sampler

$x_0$ — data latent, $x_1=\epsilon$ — noise. Строим $x_t=(1-t)x_0+t\epsilon$ и target $u=dx_t/dt=\epsilon-x_0$. Training draw может идти по t в любом порядке. **Генерация идёт t=1→0**, поэтому Euler имеет отрицательное $\Delta t$.

В части FM литературы время задаётся noise→data. Это эквивалентное соглашение после $s=1-t$, но скорость меняет знак. Нельзя взять target одного соглашения и sampler другого. [Flow Matching primary paper](https://arxiv.org/abs/2210.02747).


### A3. Визуализируем path и проверяем производную

Пока одна пара data/noise. Straight path — учебная пара, не обещание, что learned sampler по неизвестному noise попадёт именно в этот data sample. Все пары смешиваются в обучении, и сеть не знает $x_0$ напрямую.


In [ ]:
x0=torch.tensor([2.,-1.]); eps=torch.tensor([-1.,2.]); ts=torch.linspace(0,1,21)
path=(1-ts[:,None])*x0+ts[:,None]*eps
plt.plot(path[:,0],path[:,1],'.-'); plt.scatter(*x0,label='data t=0'); plt.scatter(*eps,label='noise t=1')
plt.quiver(path[::5,0],path[::5,1],*[torch.full((5,),float(v)) for v in eps-x0],angles='xy',scale_units='xy',scale=5)
plt.axis('equal'); plt.legend(); plt.show()
t=0.4; h=1e-3
finite_difference=(((1-(t+h))*x0+(t+h)*eps)-((1-(t-h))*x0+(t-h)*eps))/(2*h)
assert torch.allclose(finite_difference,eps-x0,atol=1e-3)
print('dx/dt:',finite_difference,'reverse increment has negative dt')


### A4. Почему обучение не требует ODE solver

Для каждого training sample известны $x_0$ и выбранный noise. Поэтому $x_t$ и target получаются одной формулой, без 16-step sampling. Обучаем regression:

$$L_{CFM}(\theta)=\mathbb E_{x_0,\epsilon,t,c}\|v_\theta(x_t,t,c)-(\epsilon-x_0)\|^2.$$

Математический optimum squared loss в одной точке — conditional average всех подходящих targets: $v^*(x,t,c)=\mathbb E[\epsilon-x_0\mid x_t=x,t,c]$. Поэтому **условные straight paths могут пересекаться**, а learned marginal trajectories не обязаны быть прямыми. Noise-target pair не может быть восстановлена однозначно по $x_t$.


In [ ]:
# Две разные пары пересекаются в одной точке при t=0.5.
a_data=torch.tensor([-2.]); a_noise=torch.tensor([2.])
b_data=torch.tensor([2.]); b_noise=torch.tensor([-2.])
assert torch.equal((a_data+a_noise)/2,(b_data+b_noise)/2)
targets=torch.stack([a_noise-a_data,b_noise-b_data]).flatten()
predictions=torch.linspace(-5,5,101)
loss=((predictions[:,None]-targets[None,:])**2).mean(-1)
print('targets:',targets,'best shared prediction:',predictions[loss.argmin()].item())
plt.plot(predictions,loss); plt.xlabel('shared velocity'); plt.ylabel('MSE at crossing'); plt.show()


### A5. Velocity, noise prediction и clean prediction

Для выбранного linear path $x_t=x_0+t u$. Поэтому $\hat x_0=x_t-t\hat v$ и $\hat\epsilon=x_t+(1-t)\hat v$. Velocity — не изображение и не обязательно noise prediction. Это соотношения параметризаций данного path; они не означают, что неполно обученный network выдаст точный $x_0$ за один шаг.

Проверка на известной паре позволяет поймать ошибку знака независимо от нейросети. Затем source показывает: OutLayer действительно возвращает latent channels, а sampler использует их как velocity.


In [ ]:
t=torch.tensor(0.37); xt=(1-t)*x0+t*eps; u=eps-x0
assert torch.allclose(xt-t*u,x0,atol=1e-6)
assert torch.allclose(xt+(1-t)*u,eps,atol=1e-6)
print('x0 recovered:',xt-t*u,'noise recovered:',xt+(1-t)*u)


## Часть B. Базовые опыты и связь с исходниками

### 1. Классы dit.py

TransformerEncoderBlock уточняет текст через self-attention и FFN. TransformerDecoderBlock обрабатывает visual tokens через self-attention → cross-attention → FFN. DiffusionTransformer3D связывает embeddings, RoPE, блоки и OutLayer. get_dit — factory. В README дан перечень внутренних trainable весов.


In [ ]:
source_excerpt('kandinsky/models/dit.py','class TransformerDecoderBlock',43)


### 2. Настоящая архитектура, маленькие размеры

Уменьшаем ширину и количество блоков, но сохраняем те же классы. На вход 33 канала: 16 шумовых, 16 нулевых visual conditioning, 1 нулевая маска T2V. На выход 16 каналов velocity. Такой forward позволяет проверить contract без больших весов.


In [ ]:
from labkit.native import tiny_dit
model=tiny_dit(LAB_DEVICE)
x=torch.randn(1,4,4,16,device=LAB_DEVICE)
input33=torch.cat([x,torch.zeros_like(x),torch.zeros_like(x[...,:1])],-1)
text=torch.randn(1,6,32,device=LAB_DEVICE); pooled=torch.randn(1,16,device=LAB_DEVICE)
pos=[torch.arange(1,device=LAB_DEVICE),torch.arange(2,device=LAB_DEVICE),torch.arange(2,device=LAB_DEVICE)]
with torch.no_grad(): velocity=model(input33,text,pooled,torch.tensor([500.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
print('parameters:',sum(p.numel() for p in model.parameters()),'input:',input33.shape,'output:',velocity.shape)


### 3. Увидим промежуточные формы

Forward hooks наблюдают выход, не меняя модель. Сначала visual projection выдаёт решётку [T,H/2,W/2,D]; перед блоками она превращается в [1,N,D]. OutLayer возвращает [T,H,W,16].


In [ ]:
shapes={}
def hook(name):
    def record(module,args,out):
        if isinstance(out,torch.Tensor): shapes[name]=tuple(out.shape)
    return record
handles=[module.register_forward_hook(hook(name)) for name,module in model.named_modules() if name in ['visual_embeddings','text_embeddings','visual_transformer_blocks.0','out_layer']]
with torch.no_grad(): model(input33,text,pooled,torch.tensor([500.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
for h in handles: h.remove()
print(shapes)


### 4. Flow Matching: учим направление, а не картинку

Выбираем convention, совпадающий с decreasing-time sampler: x₀ — data, ε — noise, xₜ=(1−t)x₀+tε. Производная по t равна ε−x₀. Учим сеть предсказывать эту velocity. При sampling идём t:1→0, поэтому Δt отрицательно. Маленький пример доказывает знак.


In [ ]:
data=torch.tensor([2.,-1.]); noise=torch.tensor([-0.5,0.4]); t=0.7
xt=(1-t)*data+t*noise; target=noise-data
print('mixed:',xt,'velocity:',target)
print('one exact step noise→data:',noise-target)
assert torch.allclose(noise-target,data)


### 5. Учебное обучение того же tiny DiT

Возьмём единственный фиксированный data latent. Каждый шаг рисует новые noise и t. Это намеренно простая задача: уменьшение loss показывает, что градиенты доходят до DiT, но не доказывает освоение изображений или текста. Frozen conditioning ниже — сохранённые случайные векторы, а не Qwen.


In [ ]:
model.train(); optimizer=torch.optim.Adam(model.parameters(),lr=0.002)
data_latent=torch.ones_like(x)*0.3
losses=[]
for step in range(100):
    t=torch.rand(1,device=LAB_DEVICE); noise=torch.randn_like(x)
    xt=(1-t)*data_latent+t*noise
    input33=torch.cat([xt,torch.zeros_like(xt),torch.zeros_like(xt[...,:1])],-1)
    prediction=model(input33,text.detach(),pooled.detach(),t*1000,pos,torch.arange(6,device=LAB_DEVICE))
    target=noise-data_latent
    loss=torch.nn.functional.mse_loss(prediction,target)
    optimizer.zero_grad(); loss.backward(); optimizer.step(); losses.append(loss.item())
plt.plot(losses); plt.ylabel('velocity MSE'); plt.xlabel('step'); plt.show()
print('first 10 mean:',np.mean(losses[:10]),'last 10 mean:',np.mean(losses[-10:]))


### 6. Проверяем loss на новых noise/t

Используем 20 новых пар, не делая optimizer.step. Обобщение здесь ограничено одним data latent. Следующий уровень — набор латентов и осмысленный conditioning; промышленное обучение требует данных и compute, которых этот курс не воспроизводит.


In [ ]:
model.eval(); validation=[]
with torch.no_grad():
    for _ in range(20):
        t=torch.rand(1,device=LAB_DEVICE); eps=torch.randn_like(x); xt=(1-t)*data_latent+t*eps
        inp=torch.cat([xt,torch.zeros_like(xt),torch.zeros_like(xt[...,:1])],-1)
        pred=model(inp,text,pooled,t*1000,pos,torch.arange(6,device=LAB_DEVICE))
        validation.append(torch.nn.functional.mse_loss(pred,eps-data_latent).item())
print('validation MSE:',np.mean(validation))


### 7. Структура обучаемых весов

Qwen hidden states frozen, но text_embeddings и Linguistic Token Refiner внутри DiT обучаются. Равным образом CLIP frozen, но pooled_text_embeddings обучается. RoPE частоты — buffers, не параметры. Эта граница особенно важна при LoRA.


In [ ]:
groups={}
for name,p in model.named_parameters():
    group=name.split('.')[0]; groups[group]=groups.get(group,0)+p.numel()
print(groups)
print('buffers:',[(n,tuple(b.shape)) for n,b in model.named_buffers()])


#


In [ ]:
#


## Часть C. Разложим настоящий Kandinsky forward на этапы

Выполним **те же методы**, что вызывает `DiffusionTransformer3D.forward`:

1. Qwen hidden states → Linear/LayerNorm; CLIP pooled → projection; t → sinusoidal/MLP; time+CLIP складываются.
2. Visual latent → patches; text → 1D RoPE и Linguistic Token Refiner.
3. Visual grid → flatten + 3D RoPE → CrossDiT blocks.
4. Modulated OutLayer → unpatchify → 16-channel velocity.

Время ODE t измеряется от 0 до 1, но network получает **t×1000** — это отдельный contract embeddings, не 1000 sampling steps. В input T2V есть дополнительные zero visual channels/mask. На маленьком примере сравним ручное исполнение stages с единым forward.


In [ ]:
from labkit.native import tiny_dit
trace_model=tiny_dit('cpu')
trace_x=torch.randn(1,4,4,33); trace_text=torch.randn(1,6,32); trace_clip=torch.randn(1,16)
trace_t=torch.tensor([0.5])*1000
trace_pos=[torch.arange(1),torch.arange(2),torch.arange(2)]; text_pos=torch.arange(6)
with torch.no_grad():
    te,time_emb,text_rope,ve=trace_model.before_text_transformer_blocks(trace_text,trace_t,trace_clip,trace_x,text_pos)
    print('projected text:',te.shape,'time+CLIP:',time_emb.shape,'visual grid:',ve.shape)
    for block in trace_model.text_transformer_blocks: te=block(te,time_emb,text_rope)
    ve,visual_shape,to_fractal,visual_rope=trace_model.before_visual_transformer_blocks(ve,trace_pos,(1,1,1),None)
    print('visual sequence:',ve.shape,'3D RoPE:',visual_rope.shape)
    for block in trace_model.visual_transformer_blocks: ve=block(ve,te,time_emb,visual_rope,None)
    staged=trace_model.after_blocks(ve,visual_shape,to_fractal,te,time_emb)
    direct=trace_model(trace_x,trace_text,trace_clip,trace_t,trace_pos,text_pos)
assert torch.allclose(staged,direct,atol=1e-6)
print('velocity:',direct.shape,'stage/forward difference:',(staged-direct).abs().max().item())
source_excerpt('kandinsky/models/dit.py','text_embed, time_embed, text_rope, visual_embed = self.before_text_transformer_blocks',18)


### C2. Как время и текст меняют velocity

Cross-attention вводит детальные Qwen tokens; CLIP pooled меняет modulation вместе со временем. В fresh модели modulation gates zero-initialized: некоторые residual ветви сначала могут не влиять на результат. После обучения dependence проверяется экспериментом, а не предполагается по названию attention.

На tiny DiT, обученном ранее для одного фиксированного latent, сравним ответы при разных t. Различие outputs показывает dependence; оно не доказывает точность и не показывает семантическое понимание Qwen. Случайный conditioning в учебном training не эквивалентен настоящим embeddings.


In [ ]:
probe=torch.randn_like(x)
probe_input=torch.cat([probe,torch.zeros_like(probe),torch.zeros_like(probe[...,:1])],-1)
with torch.no_grad():
    v_early=model(probe_input,text,pooled,torch.tensor([900.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
    v_late=model(probe_input,text,pooled,torch.tensor([100.],device=LAB_DEVICE),pos,torch.arange(6,device=LAB_DEVICE))
print('mean |velocity(t=.9)-velocity(t=.1)|:',(v_early-v_late).abs().mean().item())


## Часть D. Полный маленький conditional flow: учим два распределения

Дополнительная 2D задача отделяет статистический смысл FM от сложности video Transformer. Вместо DiT используем маленький MLP, но path, target и solver одинакового типа. Conditioning class=0/1 играет роль двух разных промптов. Data distribution — Gaussian clouds вокруг двух центров, noise — один стандартный Gaussian.

Обучение: случайные data/noise/t → один network forward → MSE → update. Генерация: только noise + выбранный class → повторные velocity forwards; data sample на вход не подаётся. Это принципиальное различие с реконструкцией VAE.


In [ ]:
class ToyVelocity(torch.nn.Module):
    def __init__(self):
        super().__init__(); self.net=torch.nn.Sequential(torch.nn.Linear(5,64),torch.nn.SiLU(),torch.nn.Linear(64,64),torch.nn.SiLU(),torch.nn.Linear(64,2))
    def forward(self,x,t,c):
        return self.net(torch.cat([x,t,torch.nn.functional.one_hot(c,2).float()],-1))
torch.manual_seed(12)
field=ToyVelocity(); optimizer=torch.optim.Adam(field.parameters(),lr=0.003)
centers=torch.tensor([[-2.,-0.7],[2.,0.7]]); flow_losses=[]
for step in range(650):
    classes=torch.randint(0,2,(128,)); data2=centers[classes]+0.3*torch.randn(128,2)
    noise2=torch.randn_like(data2); times=torch.rand(128,1)
    mixed=(1-times)*data2+times*noise2
    loss=torch.nn.functional.mse_loss(field(mixed,times,classes),noise2-data2)
    optimizer.zero_grad(); loss.backward(); optimizer.step(); flow_losses.append(loss.item())
plt.plot(flow_losses); plt.xlabel('training step'); plt.ylabel('CFM velocity MSE'); plt.show()


### D2. Генерируем без encoder и без data samples

Для каждого class используем одинаковый initial noise: так разница результата зависит от conditioning. Сначала сохраняем intermediate trajectory, потом сравниваем final samples с известными training distribution centers. Ошибка mean и spread — учебные проверки; они не являются image/video quality metrics.

Euler идёт по decreasing t. Uniform grid здесь выбран для ясности; Kandinsky distill grid исследуем следующим разделом. Даже если training paths прямые, sampled trajectory learned field может изгибаться.


In [ ]:
field.eval(); initial=torch.randn(300,2); generated={}; traces={}
times=torch.linspace(1,0,41)
with torch.no_grad():
    for class_id in [0,1]:
        particle=initial.clone(); labels=torch.full((len(initial),),class_id,dtype=torch.long)
        trajectory=[particle[0].clone()]
        for current,next_t in zip(times[:-1],times[1:]):
            particle=particle+(next_t-current)*field(particle,current.expand(len(particle),1),labels)
            trajectory.append(particle[0].clone())
        generated[class_id]=particle; traces[class_id]=torch.stack(trajectory)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].scatter(initial[:,0],initial[:,1],s=6,alpha=.3,label='initial noise')
for class_id,points in generated.items():
    axes[0].scatter(points[:,0],points[:,1],s=6,alpha=.5,label=f'class {class_id}')
    axes[1].plot(traces[class_id][:,0],traces[class_id][:,1],'.-',label=f'class {class_id}')
    print('class',class_id,'mean:',points.mean(0).tolist(),'target:',centers[class_id].tolist(),'std:',points.std(0).tolist())
for ax in axes: ax.set_aspect('equal'); ax.legend()
axes[0].set_title('Generated conditional distributions'); axes[1].set_title('Same noise, different conditions')
plt.tight_layout(); plt.show()


## Часть E. Что точно делает Kandinsky 5.0 inference

`get_velocity` → DiT(t×1000) → optional CFG. `generate` → shifted timesteps → visual conditioning → `img += timestep_diff * pred_velocity`. После этого latent делится на VAE scaling factor и декодируется. **Flow Matching не запускается заново при inference:** применяем обученное поле.

Для 5s distill checkpoint config содержит 16 steps и guidance=1.0: conditional velocity без второй uncond ветви. `scheduler_scale` влияет на time grid, а не на архитектуру. Ниже проверяем endpoint/monotonicity и печатаем именно source update. YAML scale=5 и pipeline default=10 расходятся; курс передаёт scale явно.


In [ ]:
from omegaconf import OmegaConf
cfg=OmegaConf.load(SOURCE/'configs/k5_lite_t2v_5s_distil_sd.yaml')
grid=shifted_schedule(cfg.model.num_steps,cfg.metrics.scheduler_scale)
assert grid[0]==1 and grid[-1]==0 and torch.all(grid.diff()<0)
print('steps:',cfg.model.num_steps,'guidance:',cfg.model.guidance_weight,'scale:',cfg.metrics.scheduler_scale)
print('first timesteps:',grid[:5],'sum dt:',grid.diff().sum().item())
source_excerpt('kandinsky/generation_utils.py','pred_velocity = dit(',15)
source_excerpt('kandinsky/generation_utils.py','img[..., :pred_velocity.shape[-1]] +=',4)


### E2. Дистилляция: почему 16 шагов — отдельные обученные веса

Distilled weights учат student эффективно воспроизводить сокращённую trajectory. На inference архитектура всё равно выдаёт velocity и sampler делает Euler updates. Один только аргумент `steps=16` не превращает SFT weights в distilled model. Снижение NFE не гарантирует меньший размер весов или latent activation peak.

Технический отчёт описывает CFG и trajectory distillation, затем post-training; этот notebook не воспроизводит полный distillation trainer. Прямой simple CFM loss выше объясняет foundation training, а не все цели distill checkpoint. [Kandinsky report, training stages](https://arxiv.org/html/2511.14993v1#S6).

### E3. Объединяем VAE и FM теорию

| Этап | Вход | Цель | Какие веса меняются |
|---|---|---|---|
| VAE training | pixels x | reconstruction NLL + KL; у real codecs возможны дополнительные terms | VAE encoder/decoder |
| DiT FM training | scaled VAE data latent, noise, t, text | velocity regression | DiT; external codec/encoders frozen в учебной схеме |
| Distillation | teacher trajectories / conditions | отдельная student objective | Student DiT |
| Generation | noise + text | forward + ODE integration, без training loss | Никакие |

VAE не «убирает Gaussian noise DiT» итерациями: он один раз кодирует/декодирует. DiT не вычисляет ELBO encoder: он учится transport в уже определённом latent space.

**Задания:**

1. Измените time convention на s=1−t, измените target и sampling direction вместе, сравните результаты.
2. Попробуйте убрать time из ToyVelocity; объясните, почему одна стрелка в x может быть недостаточна.
3. Уберите class conditioning и сравните с двумя conditional clouds.
4. Замерьте Euler 8/16/40 steps при одинаковом noise; считайте NFE отдельно от training iterations.
5. В trained tiny DiT сохраните промежуточные latents и оцените distance до known fixed data latent; сравните с random weights.
6. Проследите source: frozen Qwen → trainable LTF → cross-attention → velocity → Euler → frozen decoder.

Итог проверки: вы должны суметь объяснить каждую переменную в Euler строке upstream и воспроизвести forward через отдельные DiT stages без скрытого trainer.


### Проверьте понимание

Не меняйте сразу несколько параметров. Сначала сформулируйте гипотезу, затем измените один параметр и сравните результат с исходным. Запишите: что было входом, что стало выходом, какие размеры изменились и почему.

Учебные модели со случайными весами показывают **механизм**, но не качество Kandinsky. Тяжёлые эксперименты запускаются только при включении соответствующего флага; пропуск этих ячеек не означает успешный запуск настоящих весов.
